# Compare Stable Diffusion 1.5 base and LoRA on a free Colab T4

Select **Runtime > Change runtime type > T4 GPU** before running.

This notebook clones the repository, installs the training extras with uv, and compares
base outputs with an adapter stored in Google Drive. Train first with
`notebooks/train_colab.ipynb` or copy an existing adapter to Drive.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU")
print(torch.cuda.get_device_name(0))

## Clone and install

Replace `YOUR_GITHUB_USER` with your GitHub username or paste the full repository URL.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/YOUR_GITHUB_USER/lora-style-imagegen.git"
REPO_DIR = Path("/content/lora-style-imagegen")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!python -m pip install -q uv
!uv sync --extra train --no-dev

## Mount Google Drive

The adapter path defaults to
`MyDrive/lora-style-imagegen/outputs/lora-style/lora`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/lora-style-imagegen")
ADAPTER_DIR = DRIVE_ROOT / "outputs/lora-style/lora"
COMPARISON_DIR = DRIVE_ROOT / "outputs/comparison"
print(ADAPTER_DIR)
print(COMPARISON_DIR)

## Run the comparison

The command reads prompts, seeds, and sampling settings from the adapter metadata. Each
prompt containing the trigger also gets a matching prompt without the trigger. The
default run generates 36 images: 3 validation prompts plus 3 untriggered prompts, 3
seeds, and 2 model variants.

In [ ]:
import subprocess

subprocess.run(
    [
        "uv",
        "run",
        "compare-lora",
        "--adapter-dir",
        str(ADAPTER_DIR),
        "--output-dir",
        str(COMPARISON_DIR),
    ],
    check=True,
)

## Inspect the grid

Rows are prompts. Columns alternate base and LoRA outputs for each seed. Individual
images and `comparison_metadata.json` are written alongside the grid.

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(COMPARISON_DIR / "comparison_grid.png"), width=1400))

## Optional overrides

Add `--prompt` and `--seed` arguments to replace the defaults. Prompt templates may use
`{trigger}`. Add `--steps` or `--guidance-scale` to change sampling.

In [ ]:
import subprocess

subprocess.run(
    [
        "uv",
        "run",
        "compare-lora",
        "--adapter-dir",
        str(ADAPTER_DIR),
        "--output-dir",
        str(COMPARISON_DIR / "custom"),
        "--prompt",
        "{trigger}, a botanical study of ferns",
        "--prompt",
        "a mushroom in a forest",
        "--seed",
        "1234",
    ],
    check=True,
)